# Aula 2 — Trabalhando com coleções de dados

> **Objetivo:** guardar e percorrer conjuntos de dados usando listas e dicionários.

---

## O problema

Na aula anterior, guardamos um único registro. Mas dados chegam em lote — uma ingestão traz mil registros, um arquivo tem dez mil linhas.

Precisamos de estruturas que guardem **muitos valores de uma vez**.

## 1. Listas — sequências ordenadas

Uma lista guarda itens em ordem. Pense em uma fila de arquivos esperando para ser processados.

In [1]:
# Lista de arquivos para ingestão
arquivos = [
    'vendas_2024_01.csv',
    'vendas_2024_02.csv',
    'vendas_2024_03.csv',
    'vendas_2024_04.csv',
]

print('Total de arquivos:', len(arquivos))
print('Primeiro arquivo:', arquivos[0])   # índice começa em 0
print('Último arquivo:', arquivos[-1])    # -1 = último

Total de arquivos: 4
Primeiro arquivo: vendas_2024_01.csv
Último arquivo: vendas_2024_04.csv


**📝 O que essa célula faz:**

Cria uma **lista** chamada `arquivos`, guardando 4 nomes de arquivo em ordem, e depois acessa alguns itens dela.

```python
arquivos = ['vendas_2024_01.csv', 'vendas_2024_02.csv', ...]
```

Uma lista é criada com colchetes `[ ]`, e os itens são separados por vírgula.

**🔢 Por que `arquivos[0]` retorna o PRIMEIRO item ?**

Em Python (e na maioria das linguagens de programação), a contagem de posições **começa do zero**, não do um. Então:

| Posição (índice) | Item |
|---|---|
| `0` | `'vendas_2024_01.csv'` ← primeiro |
| `1` | `'vendas_2024_02.csv'` |
| `2` | `'vendas_2024_03.csv'` |
| `3` | `'vendas_2024_04.csv'` ← último |

**🔄 E o índice `-1`?**

Índices negativos contam **de trás para frente**: `-1` é sempre o último item, `-2` o penúltimo, e assim por diante. Isso evita ter que calcular `len(lista) - 1` toda vez que você quer o último elemento.

**📏 `len(arquivos)`** → conta quantos itens existem na lista (nesse caso, 4).

In [2]:
# Adicionando e removendo itens
arquivos.append('vendas_2024_05.csv')   # adiciona no final
print('Após append:', len(arquivos))

arquivos.remove('vendas_2024_01.csv')   # remove pelo valor
print('Após remove:', arquivos)

Após append: 5
Após remove: ['vendas_2024_02.csv', 'vendas_2024_03.csv', 'vendas_2024_04.csv', 'vendas_2024_05.csv']


**📝 O que essa célula faz:**

Modifica a lista `arquivos`, adicionando um item no final e removendo outro pelo valor.

```python
arquivos.append('vendas_2024_05.csv')   # adiciona no final
arquivos.remove('vendas_2024_01.csv')   # remove pelo nome
```

**➕ `.append(item)`** → adiciona um item **no final** da lista. Note que não tem `=` — você não escreve `arquivos = arquivos.append(...)`. O `.append()` já altera a lista original diretamente (dizemos que listas são **mutáveis**, ou seja, podem ser alteradas depois de criadas).

**➖ `.remove(valor)`** → procura pelo **valor** (não pela posição) e remove a primeira ocorrência encontrada. Se o valor não existir na lista, o Python gera um erro.

**Por que isso importa?** Em um pipeline, listas como essa frequentemente representam uma "fila de trabalho" — você adiciona arquivos novos que chegaram e remove os que já foram processados.

## 2. Percorrendo listas com `for`

O laço `for` é a principal forma de processar cada item de uma coleção:

In [3]:
camadas = ['bronze', 'silver', 'gold']

for camada in camadas:
    caminho = f's3://meu-lake/{camada}/'
    print('Verificando:', caminho)

Verificando: s3://meu-lake/bronze/
Verificando: s3://meu-lake/silver/
Verificando: s3://meu-lake/gold/


**📝 O que essa célula faz:**

Usa um laço `for` para **percorrer** cada item da lista `camadas`, um por um, executando o mesmo bloco de código para cada item.

```python
for camada in camadas:
    caminho = f's3://meu-lake/{camada}/'
    print('Verificando:', caminho)
```

**🔄 Como ler esse código em português:**

"Para cada `camada` dentro da lista `camadas`, faça: monte um caminho usando essa camada, e imprima."

A cada repetição do laço, a variável `camada` assume o valor do próximo item da lista — primeiro `'bronze'`, depois `'silver'`, depois `'gold'`. O bloco indentado (com 4 espaços, embaixo do `for`) é executado uma vez para cada item.

**Por que `for` e não copiar e colar o código 3 vezes?** Porque na vida real a lista pode ter 3 itens hoje e 300 amanhã — o `for` funciona para qualquer tamanho de lista sem precisar mudar o código.

In [3]:
# enumerate: quando você precisa do índice E do valor
arquivos_para_processar = ['log_jan.json', 'log_fev.json', 'log_mar.json']

for numero, arquivo in enumerate(arquivos_para_processar, start=1):
    print(f'[{numero}/{len(arquivos_para_processar)}] Processando: {arquivo}')

[2/3] Processando: log_jan.json
[3/3] Processando: log_fev.json
[4/3] Processando: log_mar.json


**📝 O que essa célula faz:**

Usa `enumerate()` para percorrer a lista obtendo **ao mesmo tempo** a posição (número) e o valor de cada item.

```python
for numero, arquivo in enumerate(arquivos_para_processar, start=1):
    print(f'[{numero}/{len(arquivos_para_processar)}] Processando: {arquivo}')
```

**🔢 O que `enumerate()` faz?**

Por padrão, um `for numero, item in enumerate(lista)` numera a partir de `0`. Aqui usamos `start=1` para começar a contagem a partir de `1` em vez de `0` — porque para humanos, "item 1 de 3" é mais natural de ler do que "item 0 de 3".

**📐 A f-string `f'[{numero}/{len(...)}]'`**

Monta algo como `[1/3]`, `[2/3]`, `[3/3]` — um contador de progresso. É um padrão muito comum em logs de pipeline, para acompanhar quantos itens já foram processados de um total.

## 3. Dicionários — dados com chave e valor

Um dicionário é exatamente o JSON que você já conhece: pares de chave → valor.

É a forma natural de representar um registro de dados em Python.

In [5]:
# Um registro de metadados de tabela
tabela = {
    'nome': 'clientes',
    'camada': 'silver',
    'formato': 'parquet',
    'particoes': 365,
    'tamanho_gb': 12.4,
    'ativa': True,
    'responsavel': None
}

# Acessando campos
print('Nome:', tabela['nome'])
print('Camada:', tabela['camada'])
print('Tamanho:', tabela['tamanho_gb'], 'GB')

Nome: clientes
Camada: silver
Tamanho: 12.4 GB


**📝 O que essa célula faz:**

Cria um **dicionário** chamado `tabela`, guardando vários campos relacionados a uma única tabela de dados, e acessa alguns desses campos.

```python
tabela = {
    'nome': 'clientes',
    'camada': 'silver',
    ...
}
```

**🔑 Dicionário vs Lista — qual a diferença?**

- **Lista** `[ ]` → acessa pela **posição** (`lista[0]`, `lista[1]`...)
- **Dicionário** `{ }` → acessa pelo **nome do campo** (chave), não pela posição: `tabela['nome']`

Um dicionário é exatamente a estrutura de um JSON: pares de **chave** (`'nome'`, `'camada'`...) e **valor** (`'clientes'`, `'silver'`...), separados por `:`.

**📥 `tabela['nome']`** → acessa o valor guardado na chave `'nome'`. Se a chave não existir, o Python gera um erro — diferente do `.get()` que vemos na próxima célula.

In [6]:
# .get() — acesso seguro (não quebra se a chave não existir)
responsavel = tabela.get('responsavel', 'não definido')
print('Responsável:', responsavel)

owner = tabela.get('owner', 'sem owner')
print('Owner:', owner)   # chave que não existe

Responsável: None
Owner: sem owner


**📝 O que essa célula faz:**

Mostra a diferença entre acessar um dicionário direto (`[ ]`) e de forma **segura** com `.get()`.

```python
responsavel = tabela.get('responsavel', 'não definido')
owner = tabela.get('owner', 'sem owner')
```

**🛡️ Por que usar `.get()` em vez de `tabela['chave']`?**

Se você tentasse fazer `tabela['owner']` e a chave `'owner'` não existisse no dicionário, o Python pararia o programa com um erro (`KeyError`). Já o `.get('owner', 'sem owner')` diz: "procure a chave `owner`; se não existir, **não dê erro**, apenas devolva `'sem owner'` no lugar."

**O segundo argumento do `.get()` é o valor padrão** — o que devolver caso a chave não exista. Isso é essencial em dados reais, onde campos opcionais frequentemente estão ausentes em alguns registros e presentes em outros.

In [10]:
# Adicionando e modificando campos
tabela['responsavel'] = 'time-engenharia'
tabela['ultima_atualizacao'] = '2024-01-15'

print(tabela)

{'nome': 'clientes', 'camada': 'silver', 'formato': 'parquet', 'particoes': 365, 'tamanho_gb': 12.4, 'ativa': True, 'responsavel': 'time-engenharia', 'ultima_atualizacao': '2024-01-15'}


**📝 O que essa célula faz:**

Adiciona e modifica campos de um dicionário já existente.

```python
tabela['responsavel'] = 'time-engenharia'
tabela['ultima_atualizacao'] = '2024-01-15'
```

**✏️ Como funciona:**

Se a chave **já existe** (como `'responsavel'`, que tinha valor `None`), essa linha **substitui** o valor antigo pelo novo. Se a chave **não existe ainda** (como `'ultima_atualizacao'`), essa linha **cria** o campo novo no dicionário. É a mesma sintaxe para os dois casos — o Python decide automaticamente se é criação ou atualização, dependendo se a chave já existia.

## 4. Lista de dicionários — o formato mais comum em pipelines

Quase sempre você vai trabalhar com uma **lista de dicionários**: cada dicionário é um registro, a lista é o lote.

In [12]:
# Simulando registros de eventos de ingestão
eventos = [
    {'id': 1, 'origem': 'api_pedidos',  'registros': 15_420, 'status': 'sucesso'},
    {'id': 2, 'origem': 'ftp_clientes', 'registros': 8_900,  'status': 'sucesso'},
    {'id': 3, 'origem': 'api_estoque',  'registros': 0,      'status': 'falha'},
    {'id': 4, 'origem': 'db_legado',    'registros': 3_200,  'status': 'sucesso'},
]

for evento in eventos:
    print(f"ID {evento['id']} | {evento['origem']:15} | {evento['registros']:>6} registros | {evento['status']}")

ID 1 | api_pedidos     |  15420 registros | sucesso
ID 2 | ftp_clientes    |   8900 registros | sucesso
ID 3 | api_estoque     |      0 registros | falha
ID 4 | db_legado       |   3200 registros | sucesso


**📝 O que essa célula faz:**

Cria uma **lista de dicionários** — cada item da lista é um dicionário completo representando um "evento" de ingestão. Depois percorre essa lista com `for` e imprime cada evento formatado.

```python
eventos = [
    {'id': 1, 'origem': 'api_pedidos', 'registros': 15_420, 'status': 'sucesso'},
    ...
]
```

**🧩 Por que essa é a estrutura mais comum em dados?**

Pense assim: cada **dicionário** é uma linha de uma tabela, e a **lista** é a tabela inteira. É exatamente como vem um JSON de uma API que retorna múltiplos registros.

**🎨 A formatação na f-string:** `{evento['origem']:15}` e `{evento['registros']:>6}`

Esses números depois dos dois pontos controlam o **espaçamento**:
- `:15` → reserva 15 caracteres de largura para o texto (alinhado à esquerda por padrão), fazendo as colunas ficarem alinhadas visualmente
- `:>6` → reserva 6 caracteres, alinhados à **direita** (o `>` indica a direção do alinhamento) — útil para números, que ficam mais legíveis alinhados pela direita

Isso é só estética para deixar a saída no terminal parecida com uma tabela.

In [12]:
# Calculando totais a partir de uma lista de dicionários
total_registros = 0
for evento in eventos:
    total_registros = total_registros + evento['registros']

print('Total de registros ingeridos:', total_registros)

# Forma mais compacta com sum()
total = sum(evento['registros'] for evento in eventos)
print('Total (sum):', total)

Total de registros ingeridos: 27520
Total (sum): 27520


**📝 O que essa célula faz:**

Soma o campo `registros` de todos os eventos da lista, de duas formas diferentes que dão o mesmo resultado.

**Forma 1 — manual, com `for`:**
```python
total_registros = 0
for evento in eventos:
    total_registros = total_registros + evento['registros']
```
Começa com `total_registros` valendo `0`, e a cada repetição do laço **soma** o valor do registro atual ao total acumulado.

**Forma 2 — compacta, com `sum()`:**
```python
total = sum(evento['registros'] for evento in eventos)
```
Essa é uma **generator expression** — uma forma resumida de dizer "para cada evento na lista, me dê o valor de `registros`", e a função `sum()` soma tudo automaticamente. É o mesmo resultado da Forma 1, só escrito em uma linha.

**Por que mostrar as duas formas?** Para você entender o que está acontecendo "por debaixo do capô" (Forma 1) antes de usar a versão mais compacta e idiomática do Python (Forma 2), que é a que você verá com mais frequência em código profissional.

## 5. Filtrando com `if` dentro do `for`

In [13]:
# Separar os eventos com falha
falhas = []
sucessos = []

for evento in eventos:
    if evento['status'] == 'falha':
        falhas.append(evento)
    else:
        sucessos.append(evento)

print(f'Sucessos: {len(sucessos)} | Falhas: {len(falhas)}')
print('Eventos com falha:', falhas)

Sucessos: 3 | Falhas: 1
Eventos com falha: [{'id': 3, 'origem': 'api_estoque', 'registros': 0, 'status': 'falha'}]


**📝 O que essa célula faz:**

Separa a lista de eventos em duas listas novas — `sucessos` e `falhas` — usando um `if` **dentro** do `for`.

```python
falhas = []
sucessos = []

for evento in eventos:
    if evento['status'] == 'falha':
        falhas.append(evento)
    else:
        sucessos.append(evento)
```

**🔀 Como ler esse código:**

"Comece com duas listas vazias. Para cada evento, verifique o status: se for `'falha'`, coloque na lista `falhas`; caso contrário (`else`), coloque na lista `sucessos`."

Esse é o padrão de **filtro manual**: percorrer uma coleção e decidir, item por item, em qual "caixa" (lista) cada um deve ir. É exatamente isso que um pipeline de validação de dados faz — separar registros bons dos problemáticos antes de seguir para a próxima etapa.

---

## 🏋️ Exercício — Processando um lote de ingestões

Abaixo está um lote de eventos de ingestão. Você precisa:

1. Contar quantos eventos têm status `'sucesso'`
2. Somar o total de registros processados com sucesso
3. Listar os nomes das origens que falharam
4. Calcular a taxa de sucesso em porcentagem

In [ ]:
lote = [
    {'origem': 'crm',         'registros': 22_100, 'status': 'sucesso'},
    {'origem': 'erp',         'registros': 8_450,  'status': 'sucesso'},
    {'origem': 'api_pagto',   'registros': 0,      'status': 'falha'},
    {'origem': 'app_mobile',  'registros': 54_300, 'status': 'sucesso'},
    {'origem': 'parceiro_x',  'registros': 0,      'status': 'falha'},
    {'origem': 'iot_sensores','registros': 180_000,'status': 'sucesso'},
]

# --- Seu código abaixo ---

# 1. Contagem de sucessos

# 2. Total de registros com sucesso

# 3. Origens com falha

# 4. Taxa de sucesso (%)


---

## Resumo da aula

| Conceito | Sintaxe |
|---|---|
| Criar lista | `[item1, item2, item3]` |
| Acessar item | `lista[0]`, `lista[-1]` |
| Tamanho | `len(lista)` |
| Adicionar | `lista.append(item)` |
| Criar dicionário | `{'chave': valor}` |
| Acessar campo | `dicionario['chave']` |
| Acesso seguro | `dicionario.get('chave', padrao)` |
| Percorrer | `for item in lista:` |
| Com índice | `for i, item in enumerate(lista):` |

**Próxima aula:** condições e validação de dados.